In [ ]:
import pickle as pkl

In [ ]:
with open('mydata/network_porto/porto_edges_new_simplify.pkl', 'rb') as f:
    edgeinfo = pkl.load(f)
with open('mydata/network_porto/porto_nodes_new.pkl', 'rb') as f:
    nodeinfo = pkl.load(f)

In [ ]:
print(edgeinfo[0])
print(nodeinfo['25503936'])

In [ ]:
import networkx as nx

G = nx.DiGraph()

# ---- add nodes ----
for nid, (lon, lat, _) in nodeinfo.items():
    G.add_node(
        nid,
        lon=lon,
        lat=lat
    )

# ---- add edges ----
for highway, length, u, v in edgeinfo.values():
    G.add_edge(
        u, v,
        highway=highway,
        length=length
    )

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

# assume G is your graph (undirected or directed)
# nodes have attributes: lon, lat

pos = {
    n: (G.nodes[n]["lon"], G.nodes[n]["lat"])
    for n in G.nodes
}

plt.figure(figsize=(10, 10))
nx.draw(
    G,
    pos=pos,
    node_size=1,
    edge_color="gray",
    width=0.3,
    with_labels=False
)
plt.title("Porto Road Graph")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

In [ ]:
import networkx as nx
import osmnx as ox

# convert to MultiGraph (required by osmnx)
G = nx.MultiGraph(G)

# set CRS
G.graph["crs"] = "EPSG:4326"

# add x/y attributes
for n in G.nodes:
    G.nodes[n]["x"] = G.nodes[n]["lon"]
    G.nodes[n]["y"] = G.nodes[n]["lat"]

fig, ax = ox.plot_graph(
    G,
    node_size=0,
    edge_linewidth=0.6,
    bgcolor="white"
)

In [ ]:
from collections import Counter

highway_counter = Counter(
    data.get("highway")
    for _, _, _, data in G.edges(keys=True, data=True)
)

print(highway_counter)

In [ ]:
highway = {'living_street':1, 'motorway':2, 'motorway_link':3, 'plannned':4, 'trunk':5, "secondary":6, "trunk_link":7, "tertiary_link":8, "primary":9, "residential":10, "primary_link":11, "unclassified":12, "tertiary":13, "secondary_link":14}

In [ ]:
graph_highways = set(highway_counter.keys())
mapped_highways = set(highway.keys())

missing = graph_highways - mapped_highways
unused = mapped_highways - graph_highways

print("❌ In graph but NOT in mapping:", missing)
print("⚠️ In mapping but NOT in graph:", unused)

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

highway_types = sorted(highway_counter.keys())

cmap = cm.get_cmap("tab20", len(highway_types))  # good categorical colormap

highway_color_map = {
    hwy: mcolors.to_hex(cmap(i))
    for i, hwy in enumerate(highway_types)
}

highway_color_map

In [ ]:
edge_colors = [
    highway_color_map.get(data.get("highway"), "#bbbbbb")
    for _, _, _, data in G.edges(keys=True, data=True)
]

In [ ]:
import osmnx as ox

fig, ax = ox.plot_graph(
    G,
    node_size=0,
    edge_color=edge_colors,
    edge_linewidth=0.8,
    bgcolor="white"
)

In [ ]:
import matplotlib.patches as mpatches

legend_patches = [
    mpatches.Patch(color=color, label=hwy)
    for hwy, color in highway_color_map.items()
]

ax.legend(
    handles=legend_patches,
    loc="lower left",
    fontsize=8,
    frameon=False
)

In [ ]:
import osmnx as ox
import matplotlib.patches as mpatches

# 1. Plot the graph FIRST
fig, ax = ox.plot_graph(
    G,
    node_size=0,
    edge_color=edge_colors,   # from your highway_color_map
    edge_linewidth=0.8,
    bgcolor="white",
    show=False,               # important: let us modify ax
    close=False
)

# 2. Build legend patches
legend_patches = [
    mpatches.Patch(color=color, label=hwy)
    for hwy, color in highway_color_map.items()
]

# 3. Add legend to the SAME ax
ax.legend(
    handles=legend_patches,
    loc="lower left",
    fontsize=8,
    frameon=False,
    title="Highway type"
)

# 4. Finally show
import matplotlib.pyplot as plt
plt.show()